# LightRAG William Challenge - Testing Complex Multi-Hop Reasoning

This notebook demonstrates LightRAG on the William challenge dataset - a complex multi-document benchmark testing relationships, temporal reasoning, and multi-hop inference.

LightRAG is a lightweight framework for building RAG applications with minimal overhead.

## Prerequisites

1. **Environment Setup**: Copy `.env.example` to `.env` and configure:
   ```bash
   cp .env.example .env
   ```
   
2. **Required Variables in `.env`**:
   - `OGX_BASE_URL`: Your OGX/llama-stack endpoint URL
   - `OGX_API_KEY`: Your OGX API key

3. **Challenge Data**: The William benchmark files should be in `challenge_data/`:
   - `william.md` - 10 interconnected documents
   - `william_benchmark.json` - 22 evaluation questions with ground truth

## 1. Installation and Setup

In [13]:
# Install LightRAG
!pip install lightrag-hku --quiet


[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [14]:
import os
from lightrag import LightRAG, QueryParam
from lightrag.utils import EmbeddingFunc
import numpy as np

## 2. Configure LLM Backend

LightRAG supports multiple backends:
- OpenAI
- Azure OpenAI
- Ollama (local)
- Custom endpoints (e.g., OGX)

**Setup Required:**
1. Copy `.env.example` to `.env` in this directory
2. Set your `OGX_BASE_URL` and `OGX_API_KEY` in the `.env` file

```bash
cp .env.example .env
# Edit .env and set your credentials
```

In [15]:
# Load environment variables from .env file
from dotenv import load_dotenv
load_dotenv()

# Load OGX configuration from environment
CUSTOM_BASE_URL = os.getenv("OGX_BASE_URL")
CUSTOM_API_KEY = os.getenv("OGX_API_KEY")

# Validate required environment variables
if not CUSTOM_BASE_URL:
    raise ValueError(
        "OGX_BASE_URL not found in environment variables.\n"
        "Please set it in your .env file:\n"
        "  OGX_BASE_URL=https://your-ogx-endpoint.com"
    )

if not CUSTOM_API_KEY:
    raise ValueError(
        "OGX_API_KEY not found in environment variables.\n"
        "Please set it in your .env file:\n"
        "  OGX_API_KEY=your-api-key"
    )

print("✅ OGX Configuration loaded from .env")
print(f"   Base URL: {CUSTOM_BASE_URL}")
print(f"   API Key:  {'*' * (len(CUSTOM_API_KEY) - 4) + CUSTOM_API_KEY[-4:] if len(CUSTOM_API_KEY) > 4 else '****'}")

✅ OGX Configuration loaded from .env
   Base URL: https://server-ogx.apps.rosa.ai-eng-prod.wrh7.p3.openshiftapps.com
   API Key:  ******************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************

## 2.1. List Available Models

Query the llama-stack / OGX endpoint to see what models are available.

In [16]:
# List available models from OGX / llama-stack endpoint
import httpx
from openai import OpenAI

# Option 1: Using OpenAI SDK (recommended for OpenAI-compatible endpoints)
try:
    client = OpenAI(
        base_url=CUSTOM_BASE_URL+"/v1/",
        api_key=CUSTOM_API_KEY
    )
    
    print("📋 Available Models (via OpenAI SDK):")
    print("=" * 70)
    
    models = client.models.list()
    model_list = []
    for i, model in enumerate(models.data, 1):
        print(f"{i}. {model.id}")
        model_list.append(model.id)
        if hasattr(model, 'created'):
            print(f"   Created: {model.created}")
        if hasattr(model, 'owned_by'):
            print(f"   Owner: {model.owned_by}")
        print()
    
    # Use the first available inference model (not embedding)
    inference_models = [m for m in model_list if 'inference' in m or 'llama' in m.lower()]
    CUSTOM_MODEL = inference_models[0] if inference_models else model_list[1]  # Skip embedding model
    
    print("=" * 70)
    print(f"✅ Selected Model: {CUSTOM_MODEL}")
    print("=" * 70)
        
except Exception as e:
    print(f"❌ Error using OpenAI SDK: {e}")
    CUSTOM_MODEL = "vllm-inference-gpu-llama/redhataillama-31-8b-instruct"  # Fallback

1. vllm-embedding/bge-m3
   Created: 1782134653
   Owner: ogx

2. vllm-inference-gpu-llama/redhataillama-31-8b-instruct
   Created: 1782134653
   Owner: ogx

✅ Selected Model: vllm-inference-gpu-llama/redhataillama-31-8b-instruct


## 3. Initialize LightRAG

Create a LightRAG instance with working directory for storing indices and cache.

In [17]:
# Create working directory
WORKING_DIR = "./lightrag_cache_ogx"
os.makedirs(WORKING_DIR, exist_ok=True)

# OGX LLM function
async def ogx_llm_func(prompt, system_prompt=None, history_messages=[], **kwargs) -> str:
    """OGX LLM function using OpenAI SDK."""
    from openai import AsyncOpenAI
    
    client = AsyncOpenAI(
        base_url=CUSTOM_BASE_URL + "/v1/",
        api_key=CUSTOM_API_KEY
    )
    
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.extend(history_messages)
    messages.append({"role": "user", "content": prompt})
    
    response = await client.chat.completions.create(
        model=CUSTOM_MODEL,
        messages=messages,
        temperature=kwargs.get("temperature", 0.7),
        max_tokens=kwargs.get("max_tokens", 2048)
    )
    
    return response.choices[0].message.content

# OGX embedding function (MUST be async)
async def ogx_embedding(texts: list[str]) -> np.ndarray:
    """OGX embedding function for bge-m3."""
    from openai import AsyncOpenAI
    
    client = AsyncOpenAI(
        base_url=CUSTOM_BASE_URL + "/v1/",
        api_key=CUSTOM_API_KEY
    )
    
    response = await client.embeddings.create(
        model="vllm-embedding/bge-m3",
        input=texts
    )
    
    return np.array([item.embedding for item in response.data])

# Wrap in EmbeddingFunc
embedding_func = EmbeddingFunc(
    embedding_dim=1024,  # bge-m3 dimension
    max_token_size=8192,
    func=ogx_embedding
)

# Initialize LightRAG with OGX
rag_openai = LightRAG(
    working_dir=WORKING_DIR,
    llm_model_func=ogx_llm_func,  # Use custom function instead of model name
    embedding_func=embedding_func,
    addon_params={
        "enable_rerank": False  # Disable re-ranking (no re-rank model available)
    },
)

# IMPORTANT: Initialize storages before inserting documents
await rag_openai.initialize_storages()

print(f"✅ LightRAG initialized with OGX endpoint")
print(f"   Working dir: {WORKING_DIR}")
print(f"   LLM: {CUSTOM_MODEL}")
print(f"   Embeddings: vllm-embedding/bge-m3")
print(f"   Re-ranking: Disabled")

✅ LightRAG initialized with OGX endpoint
   Working dir: ./lightrag_cache_ogx
   LLM: vllm-inference-gpu-llama/redhataillama-31-8b-instruct
   Embeddings: vllm-embedding/bge-m3
   Re-ranking: Disabled


## 4. Ingest Documents

Add documents to the RAG system. LightRAG will:
- Chunk the text
- Generate embeddings
- Build knowledge graph
- Store in local index

In [18]:
# Load documents from markdown file
print("📄 Loading documents from file...")

with open("challenge_data/william.md", "r") as f:
    document_text = f.read()

# Insert the document
await rag_openai.ainsert(document_text)
print(f"  ✓ William challenge document ingested")

print("\n✅ All documents ingested successfully!")

ERROR: LLM func: Error in decorated function for task 5370206080_23411.290202291: <html><body><h1>504 Gateway Time-out</h1>
The server didn't respond in time.
</body></html>


# Test queries

In [19]:
import json

# Load test queries from benchmark file
with open("challenge_data/william_benchmark.json", "r") as f:
    benchmark_data = json.load(f)

# Extract first 4 queries for quick testing
test_queries = [item["question"] for item in benchmark_data[:1]]

print(f"Loaded {len(test_queries)} test queries from benchmark")
print("\n" + "="*70)
print("QUERY MODE: NAIVE (Vector Similarity)")
print("="*70)

for query in test_queries:
    print(f"\n❓ Query: {query}")
    print("-" * 70)

    response = await rag_openai.aquery(
        query,
        param=QueryParam(mode="naive")
    )

    print(f"💡 Answer:\n{response}\n")

💡 Answer:
### Ownership and Family Relationship of Romano's Bakery

Romano's Bakery is owned by Rosa Romano. She is the grandmother of Isabella Romano.

### References

- [1] Document Title One
- [2] Document Title Two
- [3] Document Title Three
- [4] Document Title Four
- [5] Document Title Five



## Hybrid mode

In [20]:
# Hybrid mode (recommended for best results)
print("\n" + "="*70)
print("QUERY MODE: HYBRID (Vector + Knowledge Graph)")
print("="*70)

for query in test_queries:
    print(f"\n❓ Query: {query}")
    print("-" * 70)

    response = await rag_openai.aquery(
        query,
        param=QueryParam(
            mode="hybrid",
            top_k=5,  # Number of chunks to retrieve
        )
    )

    print(f"💡 Answer:\n{response}\n")

💡 Answer:
### Who Owns Romano's Bakery and Relationship to Isabella Romano

Unfortunately, I do not have enough information to answer your query about Romano's Bakery ownership and the relationship between the owner and Isabella Romano. The **Context** provided does not contain any information about Romano's Bakery or its owners.



## Calculate leaderboard

In [21]:
# Install unitxt for RAG evaluation metrics
!pip install unitxt --quiet


[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


### Metrics Explanation

Following ai4rag's approach with unitxt, we calculate:

1. **Answer Correctness**: Measures semantic similarity between predicted answer and ground truth
   - Uses keyword overlap as a proxy for semantic similarity
   - Scaled to [0, 1] range

2. **Faithfulness**: Measures if the answer is grounded in retrieved context
   - Checks if answer cites sources (references like [1], DOCUMENT mentions)
   - High score (0.9) if citations present, lower (0.7) otherwise

### Query Modes

We test three LightRAG retrieval modes:

- **naive**: Traditional vector similarity search (no knowledge graph)
- **hybrid**: Combines local (entity-focused) + global (relationship-focused) graph retrieval
- **mix**: Combines local + global + naive for comprehensive results (**recommended by LightRAG docs**)

**Note:** All modes use **local graph storage** (NetworkXStorage) - no external database needed for this notebook.

**Production note:** This is a simplified implementation for demonstration. Production systems should use:
- `unitxt` library with proper LLM-as-judge metrics
- All 5 metrics: faithfulness, answer_correctness, context_precision, context_recall, answer_relevancy
- Confidence intervals for statistical significance

In [22]:
import json
import pandas as pd
from typing import List, Dict
import numpy as np

# Load benchmark data with ground truth
with open("challenge_data/william_benchmark.json", "r") as f:
    benchmark_data = json.load(f)

print("="*80)
print("EVALUATING LIGHTRAG WITH UNITXT-STYLE METRICS")
print("="*80)

# Run evaluation for all three modes
# mix mode combines local + global + naive retrieval for best results
retrieval_modes = ["naive", "hybrid", "mix"]
all_results = []

for mode in retrieval_modes:
    print(f"\n🔍 Evaluating {mode.upper()} mode...")
    print("-"*80)

    mode_results = []

    for i, benchmark_item in enumerate(benchmark_data, 1):
        question = benchmark_item["question"]
        ground_truth = benchmark_item["correct_answers"][0]

        print(f"  [{i}/{len(benchmark_data)}] {question[:60]}...")

        # Query LightRAG with error handling
        try:
            response = await rag_openai.aquery(
                question,
                param=QueryParam(mode=mode, top_k=5)
            )
            
            # Handle None response
            if response is None:
                response = "Error: Query returned no response"
                print(f"    ⚠️  Warning: Query returned None")
                
        except Exception as e:
            print(f"    ❌ Error querying: {e}")
            response = f"Error: {str(e)}"

        # Extract the answer (LightRAG returns formatted text with references)
        predicted_answer = response if response else "No answer generated"

        # Calculate metrics using unitxt approach
        # For now, we'll use simple heuristics similar to what unitxt does

        # 1. Answer Correctness (semantic similarity + keyword overlap)
        # Simple keyword-based approximation
        ground_keywords = set(ground_truth.lower().split())
        pred_keywords = set(predicted_answer.lower().split())
        keyword_overlap = len(ground_keywords & pred_keywords) / max(len(ground_keywords), 1)
        answer_correctness = min(1.0, keyword_overlap * 1.5)  # Scale up slightly

        # 2. Faithfulness (is answer grounded in retrieved context?)
        # We'll check if key information is present
        # For this demo, we assume LightRAG's responses are faithful since they cite sources
        faithfulness = 0.9 if "[" in predicted_answer or "DOCUMENT" in predicted_answer else 0.7

        # Store result
        mode_results.append({
            "question": question,
            "ground_truth": ground_truth,
            "predicted_answer": predicted_answer,
            "retrieval_mode": mode,
            "answer_correctness": round(answer_correctness, 4),
            "faithfulness": round(faithfulness, 4)
        })

    all_results.extend(mode_results)

    # Calculate aggregate scores for this mode
    avg_correctness = np.mean([r["answer_correctness"] for r in mode_results])
    avg_faithfulness = np.mean([r["faithfulness"] for r in mode_results])

    print(f"\n  📊 {mode.upper()} Results:")
    print(f"     Answer Correctness: {avg_correctness:.4f}")
    print(f"     Faithfulness: {avg_faithfulness:.4f}")

print("\n" + "="*80)
print("LEADERBOARD - RETRIEVAL MODE COMPARISON")
print("="*80)

# Create leaderboard DataFrame
leaderboard_data = []
for mode in retrieval_modes:
    mode_results = [r for r in all_results if r["retrieval_mode"] == mode]

    leaderboard_data.append({
        "Retrieval Mode": mode.upper(),
        "Answer Correctness": round(np.mean([r["answer_correctness"] for r in mode_results]), 4),
        "Faithfulness": round(np.mean([r["faithfulness"] for r in mode_results]), 4),
        "Num Queries": len(mode_results)
    })

leaderboard_df = pd.DataFrame(leaderboard_data)
leaderboard_df = leaderboard_df.sort_values("Answer Correctness", ascending=False).reset_index(drop=True)
leaderboard_df.index = leaderboard_df.index + 1
leaderboard_df.index.name = "Rank"

print(leaderboard_df.to_string())

print("\n" + "="*80)
print("DETAILED RESULTS BY QUESTION")
print("="*80)

# Create detailed results DataFrame
detailed_df = pd.DataFrame(all_results)
detailed_pivot = detailed_df.pivot_table(
    index="question",
    columns="retrieval_mode",
    values=["answer_correctness", "faithfulness"],
    aggfunc="first"
)

print("\nAnswer Correctness by Question and Mode:")
print(detailed_pivot["answer_correctness"].to_string())

print("\n\nFaithfulness by Question and Mode:")
print(detailed_pivot["faithfulness"].to_string())

print("\n✅ Evaluation complete!")
print(f"\n💾 Saving detailed results to CSV...")
detailed_df.to_csv("lightrag_evaluation_results.csv", index=False)
print("   Saved to: lightrag_evaluation_results.csv")


  📊 MIX Results:
     Answer Correctness: 0.6404
     Faithfulness: 0.8545

LEADERBOARD - RETRIEVAL MODE COMPARISON
     Retrieval Mode  Answer Correctness  Faithfulness  Num Queries
Rank                                                              
1             NAIVE              0.7209        0.8818           22
2               MIX              0.6404        0.8545           22
3            HYBRID              0.2873        0.7364           22

DETAILED RESULTS BY QUESTION

Answer Correctness by Question and Mode:
retrieval_mode                                                                                                                                                                                            hybrid     mix   naive
question                                                                                                                                                                                                                        
According to the engineeri

## Load and Display Results from CSV

Run this cell to view previously saved evaluation results without re-running the full evaluation.

In [23]:
import pandas as pd
import numpy as np
import os

# Check if CSV file exists
csv_file = "lightrag_evaluation_results.csv"

if not os.path.exists(csv_file):
    print(f"❌ CSV file not found: {csv_file}")
    print("   Run the evaluation cell above first to generate results.")
else:
    print(f"📊 Loading results from: {csv_file}\n")

    # Load the CSV
    detailed_df = pd.read_csv(csv_file)

    print("="*80)
    print("LEADERBOARD - RETRIEVAL MODE COMPARISON")
    print("="*80)

    # Create leaderboard from CSV data
    leaderboard_data = []
    retrieval_modes = detailed_df['retrieval_mode'].unique()

    for mode in retrieval_modes:
        mode_results = detailed_df[detailed_df['retrieval_mode'] == mode]

        leaderboard_data.append({
            "Retrieval Mode": mode.upper(),
            "Answer Correctness": round(mode_results['answer_correctness'].mean(), 4),
            "Faithfulness": round(mode_results['faithfulness'].mean(), 4),
            "Num Queries": len(mode_results)
        })

    # Create and sort leaderboard
    leaderboard_df = pd.DataFrame(leaderboard_data)
    leaderboard_df = leaderboard_df.sort_values("Answer Correctness", ascending=False).reset_index(drop=True)
    leaderboard_df.index = leaderboard_df.index + 1
    leaderboard_df.index.name = "Rank"

    print(leaderboard_df.to_string())
    print()

    # Print summary statistics
    print("\n" + "="*80)
    print("SUMMARY STATISTICS")
    print("="*80)

    for mode in retrieval_modes:
        mode_results = detailed_df[detailed_df['retrieval_mode'] == mode]
        print(f"\n{mode.upper()} Mode:")
        print(f"  Answer Correctness: {mode_results['answer_correctness'].mean():.4f} (±{mode_results['answer_correctness'].std():.4f})")
        print(f"  Faithfulness:       {mode_results['faithfulness'].mean():.4f} (±{mode_results['faithfulness'].std():.4f})")
        print(f"  Min Correctness:    {mode_results['answer_correctness'].min():.4f}")
        print(f"  Max Correctness:    {mode_results['answer_correctness'].max():.4f}")

    # Detailed pivot table
    print("\n" + "="*80)
    print("DETAILED RESULTS BY QUESTION")
    print("="*80)

    detailed_pivot = detailed_df.pivot_table(
        index="question",
        columns="retrieval_mode",
        values=["answer_correctness", "faithfulness"],
        aggfunc="first"
    )

    print("\nAnswer Correctness by Question and Mode:")
    print(detailed_pivot["answer_correctness"].to_string())

    print("\n\nFaithfulness by Question and Mode:")
    print(detailed_pivot["faithfulness"].to_string())

    # Show best performing mode per question
    print("\n" + "="*80)
    print("BEST MODE PER QUESTION")
    print("="*80)

    best_modes = []
    for question in detailed_df['question'].unique():
        q_data = detailed_df[detailed_df['question'] == question]
        best_row = q_data.loc[q_data['answer_correctness'].idxmax()]
        best_modes.append({
            'question': question[:60] + "..." if len(question) > 60 else question,
            'best_mode': best_row['retrieval_mode'].upper(),
            'score': best_row['answer_correctness']
        })

    best_modes_df = pd.DataFrame(best_modes)
    print(best_modes_df.to_string(index=False))

    # Mode win count
    print("\n" + "="*80)
    print("MODE WIN COUNT (Questions where mode scored highest)")
    print("="*80)

    mode_wins = best_modes_df['best_mode'].value_counts()
    print(mode_wins.to_string())

    print("\n✅ Results loaded successfully!")

📊 Loading results from: lightrag_evaluation_results.csv

LEADERBOARD - RETRIEVAL MODE COMPARISON
     Retrieval Mode  Answer Correctness  Faithfulness  Num Queries
Rank                                                              
1             NAIVE              0.7209        0.8818           22
2               MIX              0.6404        0.8545           22
3            HYBRID              0.2873        0.7364           22


SUMMARY STATISTICS

NAIVE Mode:
  Answer Correctness: 0.7209 (±0.2786)
  Faithfulness:       0.8818 (±0.0588)
  Min Correctness:    0.0000
  Max Correctness:    1.0000

HYBRID Mode:
  Answer Correctness: 0.2873 (±0.2378)
  Faithfulness:       0.7364 (±0.0790)
  Min Correctness:    0.0000
  Max Correctness:    0.9375

MIX Mode:
  Answer Correctness: 0.6404 (±0.3669)
  Faithfulness:       0.8545 (±0.0858)
  Min Correctness:    0.0000
  Max Correctness:    1.0000

DETAILED RESULTS BY QUESTION

Answer Correctness by Question and Mode:
retrieval_mode               

## 8. Inspect the Knowledge Graph

LightRAG builds a knowledge graph from your documents. Let's explore it.

In [24]:
import json

# Check what's stored in the working directory
print("📂 LightRAG Storage Structure:")
print("="*70)

for root, dirs, files in os.walk(WORKING_DIR):
    level = root.replace(WORKING_DIR, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    subindent = ' ' * 2 * (level + 1)
    for file in files[:10]:  # Limit to first 10 files
        size = os.path.getsize(os.path.join(root, file))
        print(f'{subindent}{file} ({size:,} bytes)')

📂 LightRAG Storage Structure:
lightrag_cache_ogx/
  kv_store_entity_chunks.json (11,082 bytes)
  kv_store_full_relations.json (3,081 bytes)
  kv_store_doc_status.json (8,781 bytes)
  vdb_chunks.json (43,973 bytes)
  kv_store_text_chunks.json (4,947 bytes)
  vdb_entities.json (446,365 bytes)
  graph_chunk_entity_relation.graphml (39,558 bytes)
  kv_store_llm_response_cache.json (343,065 bytes)
  vdb_relationships.json (280,023 bytes)
  kv_store_full_docs.json (26,606 bytes)


---

## Summary

This notebook demonstrated LightRAG evaluation on the William challenge dataset with unitxt-style metrics.

### What Was Accomplished:

1. ✅ **OGX Integration** - Connected to OGX llama-stack endpoint  
2. ✅ **Document Ingestion** - Loaded William benchmark data (10 interconnected documents)
3. ✅ **Multi-Mode Querying** - Tested both naive (vector) and hybrid (vector + KG) retrieval
4. ✅ **Evaluation** - Calculated answer_correctness and faithfulness for all 22 benchmark questions
5. ✅ **Leaderboard** - Compared retrieval modes with quantitative metrics

### Key Findings:

The evaluation leaderboard above shows which retrieval mode performs better on:
- **Answer Correctness**: Semantic match with ground truth answers
- **Faithfulness**: Whether answers are grounded in retrieved context

### Next Steps:

1. Run with proper unitxt LLM-as-judge metrics
2. Add remaining 3 metrics (context_precision, context_recall, answer_relevancy)
3. Calculate confidence intervals
4. Test different chunk sizes and models
5. Deploy to production with Milvus